# Notebook 3: Seagrass Filler Production

**Project:** RAW Project — 3D-Printed Biopolymer Panels  
**Database:** `lca_database_3DPrintedBiopol`  
**Ecoinvent version:** 3.12, cut-off system model  
**Date:** 2026  

## Purpose

This notebook rebuilds the `seagrass filler production` activity, replacing a placeholder proxy with a literature-grounded foreground inventory. The process modelled is: collection of beached seagrass wrack (*Zostera marina*), drying, and milling into powder.

### Modelling decisions

1. **System boundary starts at collection.** The seagrass grows wild and washes ashore. No upstream production burden is assigned (cut-off approach).
2. **Wet-to-dry ratio: 3:1 (base case).** Literature values for *Zostera marina* wrack range from 2:1 to 4:1 depending on beach residence time. ⚠️ Sensitivity parameter.
3. **Drying energy: 15 MJ/kg dry (= 4.17 kWh/kg).** Drawn from Pérez-López et al. (2014) and comparable seaweed drying LCA studies (range: 10–20 MJ/kg for forced-air or belt dryers). First-principles check: ~4.5 MJ/kg theoretical minimum (2 kg water × 2.26 MJ/kg latent heat) at 3:1 ratio; ~13–15 MJ/kg at 30–35% dryer efficiency. ⚠️ Sensitivity range: 10–25 MJ/kg.
4. **Milling energy: 0.35 kWh/kg.** Proxy from bark flour filler production in this database (Wang et al. 2018, J Wood Sci 64:338–346), using the same type of cutting mill at a similar target particle size.
5. **Transport: 50 km, RER lorry.** Consistent with all other filler activities in this database.
6. **Machine wear: 0.01 kg machine/kg output.** Consistent with bark, cellulose, cotton, and wood flour filler activities. Lab-scale estimate based on Retsch SM2000 cutting mill.
7. **Electricity location: DE.** Consistent with all other activities in this database.

### Activity modified in this notebook

| Activity | Action | Notes |
|---|---|---|
| `seagrass filler production` | **Rebuild** | Replaces flax straw placeholder with literature-based inventory |

---

## 0. Setup and sanity check

In [1]:
import bw2data as bd
import bw2io as bi

# Set the active project
bd.projects.set_current("biopol_lca")

# Open databases
db = bd.Database("lca_database_3DPrintedBiopol")
ei = bd.Database("ecoinvent-3.12-cutoff")

print(f"Foreground database: {len(db)} activities")
print()
print("Activities currently in database:")
for act in sorted(db, key=lambda x: x['name']):
    exchanges = list(act.exchanges())
    inputs = [e for e in exchanges if e['type'] == 'technosphere']
    print(f"  {act['name']} ({act['location']}) — {len(inputs)} technosphere input(s)")

Foreground database: 13 activities

Activities currently in database:
  3D printing (RER) — 3 technosphere input(s)
  AEIEP pea protein isolate production (GLO) — 9 technosphere input(s)
  Drying with dehumidifying chamber (RER) — 3 technosphere input(s)
  Kiln baking - 150 degress for 2 hours (RER) — 2 technosphere input(s)
  avoided burden: nitrogen fixation, peas (GLO) — 1 technosphere input(s)
  bark flour filler production (GLO) — 4 technosphere input(s)
  cellulose filler production (GLO) — 4 technosphere input(s)
  cotton filler production (GLO) — 4 technosphere input(s)
  hemp dust filler production (GLO) — 2 technosphere input(s)
  mix preparation (RER) — 9 technosphere input(s)
  pea protein binder production (GLO) — 7 technosphere input(s)
  seagrass filler production (GLO) — 0 technosphere input(s)
  wood flour filler production (GLO) — 4 technosphere input(s)


---

## 1. Helper functions

Identical to Notebook 2. Reproduced here for self-containment.

- `find_activity`: exact name match in the foreground database.
- `find_ei`: look up an ecoinvent dataset by name, location, and optionally reference product.
- `search_ei`: broad fuzzy search across ecoinvent by keyword.
- `clear_all_technosphere`: remove every technosphere exchange from an activity.

In [2]:
def find_activity(db, name):
    """Find a single foreground activity by exact name."""
    results = [a for a in db if a['name'] == name]
    if len(results) == 0:
        raise ValueError(f"Activity not found: '{name}'")
    if len(results) > 1:
        raise ValueError(f"Multiple activities found for: '{name}'")
    return results[0]


def find_ei(name, location, ref_product=None):
    """Look up an ecoinvent activity by exact name and location."""
    results = [
        a for a in ei
        if a['name'] == name
        and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Ecoinvent dataset not found: '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} — using first")
    return results[0]


def search_ei(keyword, max_results=20):
    """
    Broad keyword search across ecoinvent. Use interactively to identify
    the correct dataset name before hard-coding it in find_ei().
    """
    keyword_lower = keyword.lower()
    results = [
        a for a in ei
        if keyword_lower in a['name'].lower()
    ]
    print(f"Found {len(results)} dataset(s) matching '{keyword}':")
    for a in results[:max_results]:
        print(f"  name:     {a['name']}")
        print(f"  location: {a['location']}")
        print(f"  ref prod: {a.get('reference product', '—')}")
        print(f"  unit:     {a.get('unit', '—')}")
        print()
    if len(results) > max_results:
        print(f"  ... and {len(results) - max_results} more. Narrow your keyword.")


def clear_all_technosphere(activity):
    """Remove every technosphere exchange from an activity."""
    to_delete = [
        exc for exc in activity.exchanges()
        if exc['type'] == 'technosphere'
    ]
    for exc in to_delete:
        exc.delete()
    print(f"  Removed {len(to_delete)} technosphere exchange(s) from '{activity['name']}'")


print("Helper functions defined.")

Helper functions defined.


---

## 2. Foreground inventory

Functional unit: **1 kg seagrass filler (dry powder)**.

Collected wrack is treated as a zero-burden raw material input — no upstream production process is modelled. Drying, milling, and transport are the only modelled processes.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Wet seagrass (collected wrack) | — | kg | Zero burden; not modelled as ecoinvent input |
| Electricity — drying | 4.17 | kWh | Pérez-López et al. (2014); range 2.78–6.94 kWh |
| Electricity — milling | 0.35 | kWh | Wang et al. (2018); proxy from bark flour |
| Machine wear | 0.01 | kg | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm | 50 km × 0.001 t/kg |

---

## 3. Retrieve background datasets

In [3]:
# ── Electricity ───────────────────────────────────────────────────────────────
ei_electricity = find_ei(
    "market for electricity, medium voltage", "DE",
    ref_product="electricity, medium voltage"
)

# ── Machine wear ──────────────────────────────────────────────────────────────
ei_machine = find_ei(
    "market for industrial machine, heavy, unspecified", "RER",
    ref_product="industrial machine, heavy, unspecified"
)

# ── Transport ─────────────────────────────────────────────────────────────────
ei_transport = find_ei(
    "market for transport, freight, lorry, unspecified", "RER",
    ref_product="transport, freight, lorry, diesel, unspecified"
)

print("Background datasets retrieved:")
print(f"  Electricity: {ei_electricity['name']} | {ei_electricity['location']}")
print(f"  Machine:     {ei_machine['name']} | {ei_machine['location']}")
print(f"  Transport:   {ei_transport['name']} | {ei_transport['location']}")

Background datasets retrieved:
  Electricity: market for electricity, medium voltage | DE
  Machine:     market for industrial machine, heavy, unspecified | RER
  Transport:   market for transport, freight, lorry, unspecified | RER


---

## 4. Modelling parameters

All numerical assumptions are defined here in one place.

In [4]:
# ── Wet-to-dry ratio ──────────────────────────────────────────────────────────
# Base case 3:1 for beached wrack (Zostera marina).
# Literature: 4:1 to 6:1 for fresh-harvested material; wrack is drier.
# ⚠️ Sensitivity parameter.
WET_TO_DRY = 3.0  # kg wet seagrass per kg dry output

# ── Drying energy ─────────────────────────────────────────────────────────────
# 15 MJ/kg dry = 4.167 kWh/kg dry.
# Source: Pérez-López et al. (2014); seaweed/seagrass drying LCA literature
# (range: 10–20 MJ/kg for forced-air or belt dryers).
# First-principles check: ~4.5 MJ/kg theoretical minimum (2 kg water evaporated
# at 3:1 ratio × 2.26 MJ/kg latent heat); ~13–15 MJ/kg at 30–35% dryer efficiency.
# ⚠️ Sensitivity range: 10–25 MJ/kg (2.78–6.94 kWh/kg).
DRYING_ENERGY_MJ = 15.0  # MJ per kg dry seagrass
DRYING_ENERGY_KWH = DRYING_ENERGY_MJ / 3.6  # kWh per kg dry seagrass

# ── Milling energy ────────────────────────────────────────────────────────────
# Proxy from bark flour filler production in this database
# (Wang et al. 2018, J Wood Sci 64:338–346); same mill type, similar particle size.
MILLING_ENERGY_KWH = 0.35  # kWh per kg dry seagrass

# ── Machine wear ──────────────────────────────────────────────────────────────
# Consistent with bark, cellulose, cotton, and wood flour filler activities.
# Lab-scale estimate: Retsch SM2000 cutting mill (~150 kg, 15-year lifespan,
# 1,000 h/year, 1 kg/h throughput). Revise to ~1e-4 to 1e-6 for industrial LCA.
MACHINE_WEAR_KG = 0.01  # kg machine per kg dry seagrass

# ── Transport ─────────────────────────────────────────────────────────────────
# Standard distance used across all filler activities in this database.
TRANSPORT_KM = 50  # km
TRANSPORT_TKM = TRANSPORT_KM * 0.001  # tkm per kg (1 kg = 0.001 t)

print("Parameters:")
print(f"  Wet-to-dry ratio:   {WET_TO_DRY}:1")
print(f"  Drying energy:      {DRYING_ENERGY_MJ} MJ/kg = {DRYING_ENERGY_KWH:.3f} kWh/kg")
print(f"  Milling energy:     {MILLING_ENERGY_KWH} kWh/kg")
print(f"  Machine wear:       {MACHINE_WEAR_KG} kg machine/kg")
print(f"  Transport:          {TRANSPORT_KM} km = {TRANSPORT_TKM} tkm/kg")

Parameters:
  Wet-to-dry ratio:   3.0:1
  Drying energy:      15.0 MJ/kg = 4.167 kWh/kg
  Milling energy:     0.35 kWh/kg
  Machine wear:       0.01 kg machine/kg
  Transport:          50 km = 0.05 tkm/kg


---

## 5. Rebuild `seagrass filler production`

The existing activity held a placeholder proxy (flax straw, dew-retted + transport). All technosphere exchanges are cleared and replaced with the inventory from Section 2.

Drying, milling, and transport are combined into a single foreground activity, consistent with all other `xxx filler production` activities in this database.

In [5]:
# ── Retrieve existing activity ─────────────────────────────────────────────────
act_seagrass = find_activity(db, "seagrass filler production")
print(f"Found: '{act_seagrass['name']}' ({act_seagrass['location']})")
print()

# ── Show current exchanges before clearing ────────────────────────────────────
print("Current exchanges (before rebuild):")
for exc in act_seagrass.exchanges():
    print(f"  [{exc['type']}] {exc.input['name']} — {exc.get('amount', '?')} {exc.input.get('unit', '')}")
print()

Found: 'seagrass filler production' (GLO)

Current exchanges (before rebuild):
  [production] seagrass filler production — 1.0 kilogram



In [6]:
# ── Clear all existing technosphere exchanges ─────────────────────────────────
clear_all_technosphere(act_seagrass)
print()

  Removed 0 technosphere exchange(s) from 'seagrass filler production'



In [7]:
# ── Add new exchanges ─────────────────────────────────────────────────────────

# 1. Electricity for drying
# 15 MJ/kg dry = 4.167 kWh/kg. Source: Pérez-López et al. (2014) and comparable
# seaweed/seagrass drying LCA studies (range: 10–20 MJ/kg, forced-air or belt dryers).
# First-principles check at 3:1 wet-to-dry ratio: ~13–15 MJ/kg at 30–35% dryer efficiency.
# ⚠️ Sensitivity range: 10–25 MJ/kg (2.78–6.94 kWh/kg).
act_seagrass.new_exchange(
    input=ei_electricity,
    amount=DRYING_ENERGY_KWH,
    type="technosphere",
    comment=(
        f"Electricity for seagrass drying: {DRYING_ENERGY_KWH:.3f} kWh/kg dry "
        f"(= {DRYING_ENERGY_MJ} MJ/kg). Source: Pérez-López et al. (2014) and "
        f"comparable seaweed/seagrass drying LCA studies (range: 10–20 MJ/kg, "
        f"forced-air or belt dryers). First-principles check at 3:1 wet-to-dry ratio: "
        f"~13–15 MJ/kg at 30–35% dryer efficiency (2 kg water evaporated × 2.26 MJ/kg "
        f"latent heat ÷ efficiency). "
        f"⚠️ Sensitivity range: 10–25 MJ/kg (2.78–6.94 kWh/kg)."
    )
).save()
print(f"  Added: electricity (drying) — {DRYING_ENERGY_KWH:.3f} kWh/kg")

# 2. Electricity for milling
# Proxy from bark flour filler production in this database
# (Wang et al. 2018, J Wood Sci 64:338–346); same mill type, similar particle size.
act_seagrass.new_exchange(
    input=ei_electricity,
    amount=MILLING_ENERGY_KWH,
    type="technosphere",
    comment=(
        f"Electricity for milling dried seagrass into powder: {MILLING_ENERGY_KWH} kWh/kg. "
        f"Proxy from bark flour filler production in this database "
        f"(Wang et al. 2018, J Wood Sci 64:338–346); same cutting mill type at similar "
        f"target particle size."
    )
).save()
print(f"  Added: electricity (milling) — {MILLING_ENERGY_KWH} kWh/kg")

# 3. Machine wear
# Consistent with bark, cellulose, cotton, and wood flour filler activities.
act_seagrass.new_exchange(
    input=ei_machine,
    amount=MACHINE_WEAR_KG,
    type="technosphere",
    comment=(
        f"Machine wear (dryer + cutting mill): {MACHINE_WEAR_KG} kg machine/kg output. "
        f"Lab-scale estimate consistent with other filler production activities in this "
        f"database. Based on Retsch SM2000 cutting mill (~150 kg, 15-year lifespan, "
        f"1,000 h/year, 1 kg/h throughput). "
        f"For industrial-scale prospective LCA, revise to ~1e-4 to 1e-6 kg/kg."
    )
).save()
print(f"  Added: machine wear — {MACHINE_WEAR_KG} kg machine/kg")

# 4. Transport
# Standard 50 km consistent with all other filler activities in this database.
act_seagrass.new_exchange(
    input=ei_transport,
    amount=TRANSPORT_TKM,
    type="technosphere",
    comment=(
        f"Transport of collected seagrass to processing facility: "
        f"{TRANSPORT_KM} km × 0.001 t/kg = {TRANSPORT_TKM} tkm/kg. "
        f"Standard distance used across all filler activities in this database."
    )
).save()
print(f"  Added: transport — {TRANSPORT_TKM} tkm/kg")

print()
print("All exchanges added.")
print()
print("Note: wet seagrass (collected wrack) is not modelled as an ecoinvent input.")
print("It is a zero-burden raw material — cut-off approach, system boundary starts at collection.")

  Added: electricity (drying) — 4.167 kWh/kg
  Added: electricity (milling) — 0.35 kWh/kg
  Added: machine wear — 0.01 kg machine/kg
  Added: transport — 0.05 tkm/kg

All exchanges added.

Note: wet seagrass (collected wrack) is not modelled as an ecoinvent input.
It is a zero-burden raw material — cut-off approach, system boundary starts at collection.


---

## 6. Verify the rebuilt activity

In [8]:
# ── Print all exchanges ────────────────────────────────────────────────────────
act_seagrass = find_activity(db, "seagrass filler production")  # refresh

print(f"Activity: '{act_seagrass['name']}'")
print(f"  Location:          {act_seagrass['location']}")
print(f"  Reference product: {act_seagrass['reference product']}")
print(f"  Unit:              {act_seagrass['unit']}")
print()
print("Exchanges:")
for exc in act_seagrass.exchanges():
    print(f"  [{exc['type']:15s}] {exc.input['name']:<55s} {exc.get('amount', '?'):>10} {exc.input.get('unit', '')}")

print()
tech_inputs = [e for e in act_seagrass.exchanges() if e['type'] == 'technosphere']
print(f"Total technosphere inputs: {len(tech_inputs)}")

Activity: 'seagrass filler production'
  Location:          GLO
  Reference product: seagrass filler
  Unit:              kilogram

Exchanges:
  [production     ] seagrass filler production                                     1.0 kilogram
  [technosphere   ] market for electricity, medium voltage                  4.166666666666667 kilowatt hour
  [technosphere   ] market for electricity, medium voltage                        0.35 kilowatt hour
  [technosphere   ] market for industrial machine, heavy, unspecified             0.01 kilogram
  [technosphere   ] market for transport, freight, lorry, unspecified             0.05 ton kilometer

Total technosphere inputs: 4


In [9]:
# ── Confirm linkage from mix preparation ──────────────────────────────────────
# seagrass filler production feeds into mix preparation as a zero-amount
# placeholder in Recipe 1. Verify the link still exists.
act_mix = find_activity(db, "mix preparation")

seagrass_links = [
    e for e in act_mix.exchanges()
    if e['type'] == 'technosphere'
    and e.input['name'] == 'seagrass filler production'
]

if seagrass_links:
    exc = seagrass_links[0]
    print(f"Link confirmed: 'mix preparation' ← 'seagrass filler production' ({exc['amount']} kg)")
    print(f"  (Amount is 0 in Recipe 1 — placeholder for optimisation runs with seagrass recipes.)")
else:
    print("WARNING: No link found from 'mix preparation' to 'seagrass filler production'.")
    print("Check that the placeholder exchange exists in mix preparation.")

Link confirmed: 'mix preparation' ← 'seagrass filler production' (0.0 kg)
  (Amount is 0 in Recipe 1 — placeholder for optimisation runs with seagrass recipes.)


---

## 7. Final database state

In [10]:
print(f"Final foreground database: {len(db)} activities")
print()
for act in sorted(db, key=lambda x: x['name']):
    tech_inputs = [e for e in act.exchanges() if e['type'] == 'technosphere']
    print(f"  {act['name']} ({act['location']}) — {len(tech_inputs)} technosphere input(s)")

Final foreground database: 13 activities

  3D printing (RER) — 3 technosphere input(s)
  AEIEP pea protein isolate production (GLO) — 9 technosphere input(s)
  Drying with dehumidifying chamber (RER) — 3 technosphere input(s)
  Kiln baking - 150 degress for 2 hours (RER) — 2 technosphere input(s)
  avoided burden: nitrogen fixation, peas (GLO) — 1 technosphere input(s)
  bark flour filler production (GLO) — 4 technosphere input(s)
  cellulose filler production (GLO) — 4 technosphere input(s)
  cotton filler production (GLO) — 4 technosphere input(s)
  hemp dust filler production (GLO) — 2 technosphere input(s)
  mix preparation (RER) — 9 technosphere input(s)
  pea protein binder production (GLO) — 7 technosphere input(s)
  seagrass filler production (GLO) — 4 technosphere input(s)
  wood flour filler production (GLO) — 4 technosphere input(s)


---

## Summary

| Activity | Action | Key exchanges |
|---|---|---|
| `seagrass filler production` | **Rebuilt** | Electricity drying (4.17 kWh), electricity milling (0.35 kWh), machine wear (0.01 kg), transport (0.05 tkm) |

### Parameters flagged for sensitivity analysis

| Parameter | Value used | Range | Source |
|---|---|---|---|
| Drying energy | 15 MJ/kg (4.17 kWh/kg) | 10–25 MJ/kg | Pérez-López et al. (2014) |
| Wet-to-dry ratio | 3:1 | 2:1–5:1 | Literature estimate for wrack |
| Milling energy | 0.35 kWh/kg | — | Proxy: Wang et al. (2018) |
| Transport distance | 50 km | — | Database standard |